# Notebook 02: Thực nghiệm phân cụm khách hàng RFM

Notebook này tiếp tục từ bảng `rfm_table.csv` được tạo ở Notebook 01. Mỗi dòng là một khách hàng; các thuộc tính hành vi được đưa vào phân cụm là `Recency`, `Frequency`, `Monetary`. `CustomerID` chỉ dùng để nhận diện khách hàng và đếm số khách trong cụm, không đưa vào tính khoảng cách.

### Luồng xử lý dữ liệu và thực nghiệm

1. Đọc bảng RFM đã làm sạch; kiểm tra đủ ba biến RFM, bỏ dòng thiếu các biến này và dừng nếu có giá trị âm vì bước log không phù hợp với dữ liệu âm.
2. Lấy tối đa 2.000 khách hàng ngẫu nhiên, không hoàn lại, với `random_state=42`. Cùng một mẫu được dùng cho mọi thuật toán để so sánh công bằng và giới hạn thời gian chạy.
3. Biến đổi `log1p` từng biến RFM nhằm nén độ lệch phải của Frequency/Monetary và giữ được giá trị 0; sau đó `StandardScaler` đưa ba biến về cùng thang đo để biến có đơn vị lớn không lấn át khoảng cách.
4. Huấn luyện baseline K-Means (`k=3`), thử K-Means với `k=2..8`, phân cụm phân cấp Ward với `k=2..8`, và DBSCAN trên các cặp `eps`/`min_samples` đã định trước.
5. Tính Silhouette trên cùng 1.000 khách hàng đầu của mẫu (hoặc toàn bộ nếu mẫu nhỏ hơn). Với DBSCAN, bỏ nhãn nhiễu `-1` chỉ khi tính Silhouette và luôn báo riêng tỷ lệ nhiễu. Chỉ số này không có đáp án đúng/sai vì đây là học không giám sát.
6. Chọn cấu hình có Silhouette cao nhất trong khoảng 2–8 cụm, ưu tiên nghiệm có không quá 20% nhiễu nếu có. Gán nhãn mô hình thắng cho mẫu, tổng hợp trung bình/trung vị RFM, so sánh trung vị cụm với trung vị của mẫu và lưu bảng, biểu đồ, nhật ký.

In [7]:
# Ket qua va dien giai EXP-034 den EXP-036 xem docs/Danh_gia_mo_hinh_EXP034_EXP036.md.
# Script src/run_clustering_experiments.py tu dong cap nhat metric da thuat toan va mo hinh chot sau khi chay.

## Chuẩn bị dữ liệu và chạy các thuật toán

Script đọc bảng RFM, lấy mẫu, biến đổi và chuẩn hóa dữ liệu, chạy các cấu hình phân cụm, chấm điểm rồi lưu bảng/biểu đồ/experiment log. Cell này thực hiện toàn bộ phần tính toán.

In [8]:
# Cell này chuẩn bị đường dẫn, xác nhận đầu vào và chạy toàn bộ thí nghiệm.
from pathlib import Path
import subprocess
import sys
import pandas as pd
from IPython.display import display, HTML

PROJECT_NAME = "Online_Retail_II_GiaiDoan1"
SCRIPT_RELATIVE = Path("src") / "run_clustering_experiments.py"
_search_roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
_candidates = [item for root in _search_roots for item in (root, root / PROJECT_NAME)]
PROJECT_ROOT = next(
    (item for item in _candidates if (item / SCRIPT_RELATIVE).is_file()), None
)

subprocess.run(
    [sys.executable, str(PROJECT_ROOT / SCRIPT_RELATIVE)],
    cwd=PROJECT_ROOT,
    check=True,
)

# Căn giữa bảng kết quả trong notebook.
display(HTML("""<style>
.dataframe { margin-left: auto !important; margin-right: auto !important; }
.dataframe th, .dataframe td { text-align: center !important; }
</style>"""))


## 1. So sánh cấu hình và thuật toán

Bảng này so sánh một baseline K-Means `k=3`, bảy cấu hình K-Means (`k=2..8`), bảy cấu hình Ward và 18 cấu hình DBSCAN trên cùng mẫu RFM đã log1p + StandardScaler. Với mỗi cấu hình, script ghi số cụm, kích thước từng cụm, noise và Silhouette. Inertia chỉ dùng so sánh các giá trị `k` trong cùng pipeline. Việc chọn scaler cuối cùng được kiểm tra riêng ở EXP-035 với cùng sample, cùng k và cùng seed.

In [9]:
TABLES = PROJECT_ROOT / 'outputs' / 'tables'
from IPython.display import Markdown
comparison = pd.read_csv(TABLES / 'clustering_model_comparison.csv')
display(Markdown('### Bảng cấu hình, số cụm, noise và Silhouette'))
display(comparison.sort_values('silhouette_excluding_noise', ascending=False, na_position='last'))

# Các chỉ số nội tại bổ sung được tính trên cùng 1.000 khách đánh giá.
multi_metrics = pd.read_csv(TABLES / 'exp036_all_algorithms_multi_metrics.csv')
metric_columns = ['algorithm', 'k', 'parameters', 'sample_n', 'evaluation_sample_n', 'silhouette', 'davies_bouldin', 'calinski_harabasz', 'noise_points', 'noise_percent', 'inertia']
display(Markdown('### EXP-036: Silhouette, Davies-Bouldin, Calinski-Harabasz và noise'))
display(multi_metrics[[column for column in metric_columns if column in multi_metrics.columns]].sort_values('silhouette', ascending=False, na_position='last'))


,model,parameters,clusters_excluding_noise,noise_points,noise_percent,silhouette_excluding_noise,inertia,cluster_sizes,evaluated_customers,evaluation_note
1,KMeans_tuned,n_clusters=2; init=k-means++; n_init=20; rando...,2,0,0.00,0.424588,2855.724477,"{""0"": 1162, ""1"": 838}",2000,Silhouette trên điểm không nhiễu; không tính đ...
3,KMeans_tuned,n_clusters=4; init=k-means++; n_init=20; rando...,4,0,0.00,0.366071,1630.053340,"{""0"": 475, ""1"": 410, ""2"": 428, ""3"": 687}",2000,Silhouette trên điểm không nhiễu; không tính đ...
0,KMeans_baseline,n_clusters=3; init=k-means++; n_init=20; rando...,3,0,0.00,0.356581,2087.309634,"{""0"": 828, ""1"": 417, ""2"": 755}",2000,Silhouette trên điểm không nhiễu; không tính đ...
2,KMeans_tuned,n_clusters=3; init=k-means++; n_init=20; rando...,3,0,0.00,0.356581,2087.309634,"{""0"": 828, ""1"": 417, ""2"": 755}",2000,Silhouette trên điểm không nhiễu; không tính đ...
4,KMeans_tuned,n_clusters=5; init=k-means++; n_init=20; rando...,5,0,0.00,0.339272,1350.838038,"{""0"": 376, ""1"": 581, ""2"": 442, ""3"": 449, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
17,DBSCAN,eps=0.25; min_samples=20; metric=euclidean,8,1288,64.40,0.335453,NaN,"{""-1"": 1288, ""0"": 344, ""1"": 49, ""2"": 162, ""3"":...",2000,Silhouette trên điểm không nhiễu; không tính đ...
8,Hierarchical_Ward,n_clusters=2; linkage=ward,2,0,0.00,0.332902,NaN,"{""0"": 1081, ""1"": 919}",2000,Silhouette trên điểm không nhiễu; không tính đ...
5,KMeans_tuned,n_clusters=6; init=k-means++; n_init=20; rando...,6,0,0.00,0.331906,1175.403456,"{""0"": 259, ""1"": 292, ""2"": 425, ""3"": 162, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
6,KMeans_tuned,n_clusters=7; init=k-means++; n_init=20; rando...,7,0,0.00,0.319175,1042.693315,"{""0"": 254, ""1"": 255, ""2"": 299, ""3"": 390, ""4"": ...",2000,Silhouette trên điểm không nhiễu; không tính đ...
23,DBSCAN,eps=0.45; min_samples=20; metric=euclidean,2,194,9.70,0.316413,NaN,"{""-1"": 194, ""0"": 510, ""1"": 1296}",2000,Silhouette trên điểm không nhiễu; không tính đ...


## 2. Mô hình được chọn theo quy tắc đã nêu

Ở EXP-036, dùng bảng trên để so K-Means/Ward/DBSCAN trong cùng pipeline StandardScaler; các nghiệm có dưới hai cụm hợp lệ không dùng để xếp hạng và noise được xét riêng. Sau đó đối chiếu scaler trong EXP-035 trên cùng 2.000 khách, cùng k và seed 42. K-Means RobustScaler k=2 đạt Silhouette 0,445749 (so với 0,443337 của StandardScaler k=2), DBI/CHI cũng nhỉnh hơn và không có noise; kiểm tra 5 seed cho ARI cặp nhỏ nhất 1,0. Vì vậy cấu hình cuối là K-Means + log1p + RobustScaler + k=2, n_init=20, random_state=42. Không so trực tiếp Inertia giữa hai scaler; kết quả chỉ áp dụng cho mẫu và cách tiền xử lý đã nêu.

In [10]:
decision = pd.read_csv(TABLES / 'model_decision_exp034_exp036.csv')
display(Markdown('### Bảng quyết định EXP-034 đến EXP-036 và các đánh đổi'))
display(decision.sort_values('silhouette', ascending=False, na_position='last'))

selected = pd.read_csv(TABLES / 'selected_clustering_model.csv')
display(Markdown('### Cấu hình cuối cùng được chọn'))
display(selected)


,model,scaler,k,init,n_init,random_state,sample_n,sampling,preprocessing,silhouette,davies_bouldin,calinski_harabasz,noise_points,noise_percent,seed_stability_seeds,seed_stability_min_pairwise_ari,seed_silhouette_range,seed_inertia_range,decision,limitations
0,KMeans,RobustScaler,2,k-means++,20,42,2000,"Fixed 2,000 CustomerID sample; random_state=42","log1p(Recency, Frequency, Monetary) -> RobustS...",0.430303,0.875147,1058.491035,0,0.0,"42,7,21,123,2026",1.0,0.0,0.0,Chọn RobustScaler k=2 theo quy tắc Silhouette ...,"Đánh giá không giám sát trên mẫu cố định, khôn..."


## 3. Hồ sơ khách hàng theo cụm: trung bình và trung vị

- Sau khi chọn cấu hình, nhãn cụm được gắn lại vào từng khách hàng trong mẫu. Nhóm theo nhãn cụm rồi tính số khách hàng, trung bình, trung vị của Recency/Frequency/Monetary và tổng Monetary. Các điểm DBSCAN có nhãn `-1` không được tính vào hồ sơ cụm vì chúng không thuộc cụm nào.

- Recency tính bằng ngày kể từ lần mua gần nhất nên thấp hơn là mua gần đây hơn; Frequency là số hóa đơn, Monetary là tổng chi tiêu (GBP), hai biến này cao hơn nghĩa là hoạt động/chi tiêu lớn hơn. Trung bình nhạy với khách hàng chi tiêu rất lớn; trung vị thường đại diện điển hình tốt hơn khi phân phối lệch. `Monetary_sum` mô tả tổng doanh thu của nhóm trong mẫu, không phải giá trị trung bình mỗi khách.

In [11]:
profiles = pd.read_csv(TABLES / 'selected_model_cluster_profiles.csv')
display(profiles)


,cluster,customers,Recency_median,Recency_mean,Frequency_median,Frequency_mean,Monetary_median,Monetary_mean,Monetary_sum
0,0,1190,282.0,294.510084,2.0,2.075630,417.760,563.926229,671072.213
1,1,810,24.0,56.969136,8.5,12.990123,3037.605,6115.905998,4953883.858


## 4. So sánh trung vị cụm với trung vị toàn mẫu

- Từ cùng mẫu 2.000 khách hàng, tính trung vị chung của mỗi biến RFM. Sau đó lấy trung vị của từng cụm so với trung vị chung và gắn nhãn “cao hơn”, “thấp hơn” hoặc “bằng”. Bảng này chuyển các con số ở phần 3 thành mô tả tương đối dễ đọc.

- Cụm 0 có Recency cao hơn nhưng Frequency và Monetary thấp hơn trung vị mẫu, gợi ý nhóm ít hoạt động gần đây và chi tiêu thấp trong mẫu. Cụm 1 có Recency thấp hơn, Frequency và Monetary cao hơn, gợi ý nhóm mua gần đây và chi tiêu nhiều hơn. Đây chỉ là mô tả của mẫu, không phải nhãn khách hàng chắc chắn hay quan hệ nhân quả.

In [12]:
interpretation = pd.read_csv(TABLES / 'cluster_interpretation.csv')
display(interpretation)

print((TABLES / 'cluster_interpretation.md').read_text(encoding='utf-8'))


,cluster,customers,Recency_median,Recency_mean,Frequency_median,Frequency_mean,Monetary_median,Monetary_mean,Monetary_sum,Recency_vs_sample_median,Frequency_vs_sample_median,Monetary_vs_sample_median
0,0,1190,282.0,294.510084,2.0,2.075630,417.760,563.926229,671072.213,cao hơn trung vị mẫu,thấp hơn trung vị mẫu,thấp hơn trung vị mẫu
1,1,810,24.0,56.969136,8.5,12.990123,3037.605,6115.905998,4953883.858,thấp hơn trung vị mẫu,cao hơn trung vị mẫu,cao hơn trung vị mẫu


# Diễn giải cụm từ mẫu RFM

Mô hình được chọn: **K-Means + log1p + RobustScaler + k=2**, `n_init=20`, `random_state=42`.
Silhouette=0.4303; Davies–Bouldin=0.8751; 
Calinski–Harabasz=1058.49; noise=0%.
Nhãn cụm được tạo lại bằng RobustScaler; phần diễn giải dưới đây dùng trung vị của cùng mẫu 2.000 khách.
Trung vị toàn mẫu: Recency=92.00 ngày, 
Frequency=3.00 hóa đơn, Monetary=915.03 GBP.

- **Cụm 0** (1,190 khách): Recency trung vị 282.00 ngày (cao hơn trung vị mẫu), Frequency 2.00 (thấp hơn trung vị mẫu), Monetary 417.76 GBP (thấp hơn trung vị mẫu); tổng Monetary 671072.21 GBP.
- **Cụm 1** (810 khách): Recency trung vị 24.00 ngày (thấp hơn trung vị mẫu), Frequency 8.50 (cao hơn trung vị mẫu), Monetary 3037.61 GBP (cao hơn trung vị mẫu); tổng Monetary 4953883.86 GBP.

## Giới hạn

Đây là mô tả trên mẫu cố định, không phải nhãn chuẩn hay bằng chứng nhân quả; cần kiểm tra ý nghĩa kinh doanh và xác nhận thêm trên toàn bộ dữ liệu.

